## Install the datasets Package

In [ ]:
!pip install -qU datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 11.9 MB/s eta 0:00:00


## Load the Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF", split="train")
print(f"Total transactions: {len(dataset):,}")

README.md:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): reconstructing file:   0%|          |  0.00B /  138MB            

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): downloading bytes:           |  0.00B            

...

Cifer-Fraud-Detection-Dataset-AF-part-8-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/21000000 [00:00<?, ? examples/s]

Total transactions: 21,000,000


In [ ]:
dataset = dataset.select(range(min(1_000_000, len(dataset))))
print(f"Working with: {len(dataset):,} transactions")

Working with: 1,000,000 transactions


## Filter Fraud and Non-Fraud

In [ ]:
fraud_data = dataset.filter(lambda x: x['isFraud'] == 1)
non_fraud_data = dataset.filter(lambda x: x['isFraud'] == 0)

print(f"Fraud transactions: {len(fraud_data):,}")
print(f"Non-fraud transactions: {len(non_fraud_data):,}")

Filter:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Fraud transactions: 1,313
Non-fraud transactions: 998,687


## Sampling the Dataset

In [ ]:
fraud_sample = fraud_data.shuffle(seed=42).select(range(500))
non_fraud_sample = non_fraud_data.shuffle(seed=42).select(range(500))

## Combine and Shuffle

In [ ]:
from datasets import concatenate_datasets

balanced_dataset = concatenate_datasets([fraud_sample, non_fraud_sample])
balanced_dataset = balanced_dataset.shuffle(seed=42)

print(f"Final training set: {len(balanced_dataset)} transactions")

Final training set: 1000 transactions


## Converting to Conversational Format

In [ ]:
def create_conversation(example):
    txn_type = example['type']
    amount = example['amount']
    old_orig = example['oldbalanceOrg']
    new_orig = example['newbalanceOrig']
    old_dest = example['oldbalanceDest']
    new_dest = example['newbalanceDest']

    user_msg = f"""Analyze this transaction for fraud risk:
- Type: {txn_type}
- Amount: ${amount:,.2f}
- Sender Balance Before: ${old_orig:,.2f}
- Sender Balance After: ${new_orig:,.2f}
- Recipient Balance Before: ${old_dest:,.2f}
- Recipient Balance After: ${new_dest:,.2f}"""

    if example['isFraud'] == 1:
        asst_msg = "HIGH"
    else:
        asst_msg = "LOW"

    return {"messages": [
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": asst_msg}
    ]}

In [ ]:
train_dataset = balanced_dataset.map(create_conversation)
print(f"Created {len(train_dataset)} training conversations")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Created 1000 training conversations


## Verify the Format

In [ ]:
print(train_dataset[0]['messages'])

[{'content': 'Analyze this transaction for fraud risk:\n- Type: CASH_IN\n- Amount: $243.74\n- Sender Balance Before: $16,101.76\n- Sender Balance After: $11,824,895.38\n- Recipient Balance Before: $1,028,631.69\n- Recipient Balance After: $8.45', 'role': 'user'}, {'content': 'LOW', 'role': 'assistant'}]


## Configure 4-bit Quantization

In [ ]:
!pip install -qU transformers bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.9 MB/s eta 0:00:00


In [ ]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

## Load the Tokenizer and Model

In [ ]:
!pip install -q accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Model loaded: {model_id}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


## Configuring LoRA Adapters

In [ ]:
!pip install -qU peft

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

## Training with SFTTrainer(supervised-fine-tuning)

In [ ]:
!pip install -qU datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.2 MB/s eta 0:00:00


In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./fraud-detector",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512
)

## Creating the Trainer

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args
)

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [19]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,2.038664
20,1.033611
30,0.853885
40,0.809396
50,0.746434
60,0.738942
70,0.746989
80,0.742240
90,0.736204
100,0.737108


Step,Training Loss
10,2.038664
20,1.033611
30,0.853885
40,0.809396
50,0.746434
60,0.738942
70,0.746989
80,0.742240
90,0.736204
100,0.737108


TrainOutput(global_step=375, training_loss=0.783994208017985, metrics={'train_runtime': 1847.1306, 'train_samples_per_second': 1.624, 'train_steps_per_second': 0.203, 'total_flos': 3218508755976192.0, 'train_loss': 0.783994208017985, 'entropy': 0.7347793459892273, 'num_tokens': 389910.0, 'mean_token_accuracy': 0.7230448544025421, 'epoch': 3.0})

## Uploading to HuggingFace Hub

In [20]:
from huggingface_hub import notebook_login

notebook_login()

In [8]:
from huggingface_hub import delete_repo, create_repo

delete_repo(repo_id=repo_name, repo_type="model")
create_repo(repo_id=repo_name, repo_type="model")

RepoUrl('https://huggingface.co/devonfire/Financial-Fraud-Detection-Model-Qwen-1.5b', endpoint='https://huggingface.co', repo_type='model', repo_id='devonfire/Financial-Fraud-Detection-Model-Qwen-1.5b')

In [9]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="/content/fraud-detector/checkpoint-375",
    repo_id=repo_name,
    repo_type="model",
    ignore_patterns=[
        "optimizer.pt", "scheduler.pt", "rng_state.pth",
        "trainer_state.json", "training_args.bin", "*.pth",
    ],
)

CommitInfo(commit_url='https://huggingface.co/devonfire/Financial-Fraud-Detection-Model-Qwen-1.5b/commit/cedce75853863ed32f29bab49f19d5e067a5b816', commit_message='Upload folder using huggingface_hub', commit_description='', oid='cedce75853863ed32f29bab49f19d5e067a5b816', pr_url=None, repo_url=RepoUrl('https://huggingface.co/devonfire/Financial-Fraud-Detection-Model-Qwen-1.5b', endpoint='https://huggingface.co', repo_type='model', repo_id='devonfire/Financial-Fraud-Detection-Model-Qwen-1.5b'), pr_revision=None, pr_num=None)

In [10]:
from huggingface_hub import list_repo_files
print(list_repo_files(repo_name))

['.gitattributes', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


## Testing the Fine-Tuned Model from Hub

In [29]:
!pip install -q --upgrade torchao

In [11]:
from transformers import pipeline

generator = pipeline("text-generation", model=repo_name, device="cuda")

adapter_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 8.75MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/224 [00:00<?, ?it/s]

In [12]:
question = """Analyze this transaction for fraud risk:
- Type: TRANSFER
- Amount: $85,000.00
- Sender Balance Before: $85,000.00
- Sender Balance After: $0.00
- Recipient Balance Before: $0.00
- Recipient Balance After: $0.00"""

output = generator([{"role": "user", "content": question}], max_new_tokens=150, return_full_text=False)[0]
print(output["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


LOW


In [13]:
question = """Analyze this transaction for fraud risk:
- Type: PAYMENT
- Amount: $150.00
- Sender Balance Before: $5,000.00
- Sender Balance After: $4,850.00
- Recipient Balance Before: $1,000.00
- Recipient Balance After: $1,150.00"""

output = generator([{"role": "user", "content": question}], max_new_tokens=150, return_full_text=False)[0]
print(output["generated_text"])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


HIGH
